# VBZ Tramlinien — Übersicht & Streckenführung 2025

Umfassende Dokumentation aller Tramlinien der Verkehrsbetriebe Zürich mit aktueller Streckenführung,
Haltestellen und interaktiven Netzwerkkarten.

**Referenz:** Offizielle GTFS-Daten (j25 / 2025) · IST-Verkehrsdaten 2023–2025 · network-map.html als Basis

In [ ]:
# Setup & Imports
import pandas as pd
import polars as pl
from pathlib import Path
import plotly.graph_objects as go

from zh_tram_flow.config import PATHS, LINE_COLORS

# GTFS Parquet-Dateien laden
gtfs_root = PATHS['raw'] / 'gtfs'
sf_gtfs_root = PATHS['root'].parent / 'sf_data-research' / 'data' / 'interim' / 'vbz' / 'gtfs'

print('Loading GTFS 2025 parquet files...')
routes_df = pl.read_parquet(gtfs_root / 'gtfs_tram_routes.parquet').to_pandas()
stops_df = pl.read_parquet(gtfs_root / 'gtfs_tram_stops.parquet').to_pandas()
trips_df = pl.read_parquet(gtfs_root / 'gtfs_tram_trips.parquet').to_pandas()
shapes_df = pl.read_parquet(gtfs_root / 'gtfs_tram_shapes.parquet').to_pandas()
stop_times_df = pl.read_parquet(sf_gtfs_root / 'gtfs_tram_stop_times.parquet').to_pandas()

print('✓ GTFS 2025 loaded')
print(f'  Stops: {len(stops_df)}')
print(f'  Routes: {len(routes_df)}')
print(f'  Trips: {len(trips_df)}')
print(f'  Stop Times: {len(stop_times_df)}')

# Filter to 2025 (year is stored as STRING '2025', not integer 2025)
routes_df = routes_df[routes_df['year'] == '2025']
trips_df = trips_df[trips_df['year'] == '2025']
stop_times_df = stop_times_df[stop_times_df['year'] == '2025']

print(f'✓ Filtered to 2025 (j25)')
print(f'  Routes: {len(routes_df)}, Trips: {len(trips_df)}, StopTimes: {len(stop_times_df)}')

In [ ]:
# Extract tram lines
tram_routes = routes_df[
    routes_df['route_short_name'].str.match(r'^\d+$|^E$')
][['route_id', 'route_short_name', 'route_long_name']].copy()

# Sort
tram_routes['sort_key'] = tram_routes['route_short_name'].map(
    lambda x: int(x) if x.isdigit() else 999
)
tram_routes = tram_routes.sort_values('sort_key').drop('sort_key', axis=1).reset_index(drop=True)

print('VBZ Tramlinien 2025:')
print(f'Total: {len(tram_routes)} Linien\n')
for _, row in tram_routes.iterrows():
    ln = row['route_short_name']
    color = LINE_COLORS.get(ln, '#999999')
    print(f'  L{ln:2s} — {row["route_long_name"]:60s} [{color}]')

## Übersicht: Alle Tramlinien mit Farben

In [ ]:
import plotly.graph_objects as go

lines = []
colors = []
names = []

for _, row in tram_routes.iterrows():
    ln = row['route_short_name']
    name = row['route_long_name']
    color = LINE_COLORS.get(ln, '#999999')
    lines.append(f'L{ln}')
    colors.append(color)
    names.append(name)

fig = go.Figure(data=[go.Bar(
    y=lines,
    marker=dict(color=colors),
    text=names,
    textposition='outside',
    orientation='h',
    hovertemplate='<b>%{y}</b><br>%{text}<extra></extra>'
)])
fig.update_layout(
    title='VBZ Tramlinien 2025 — Offizielle Farben',
    xaxis_title='',
    yaxis_title='',
    height=500,
    showlegend=False,
    margin=dict(l=50, r=300, t=60, b=50)
)
fig.show()

## Inhaltsverzeichnis

Klicke auf eine Tramlinie um Details zu sehen:

In [ ]:
# Generate TOC
toc_items = []
for _, row in tram_routes.iterrows():
    ln = row['route_short_name']
    name = row['route_long_name']
    toc_items.append(f'- **[Linie {ln}](#{ln})** — {name}')

print('\n'.join(toc_items))

---\n\n## Streckenführung pro Tramlinie\n\nJede Linie zeigt die aktuelle 2025er Streckenführung mit beiden Fahrtrichtungen, Haltestellen und interaktiver Karte.

In [ ]:
# Helper: Get line stops for a direction
def get_line_stops(line_name, direction_id='0'):
    """Extract stops for a line and direction with coordinates."""
    route = routes_df[routes_df['route_short_name'] == line_name]
    if route.empty:
        return []
    route_id = route.iloc[0]['route_id']
    
    trips = trips_df[
        (trips_df['route_id'] == route_id) &
        (trips_df['direction_id'] == direction_id)
    ]
    if trips.empty:
        return []
    trip_id = trips.iloc[0]['trip_id']
    
    trip_stops = stop_times_df[
        stop_times_df['trip_id'] == trip_id
    ].sort_values('stop_sequence')
    if trip_stops.empty:
        return []
    
    result = []
    for _, st_row in trip_stops.iterrows():
        stop_id = st_row['stop_id']
        stop_info = stops_df[stops_df['stop_id'] == stop_id]
        if not stop_info.empty:
            stop = stop_info.iloc[0]
            result.append({
                'name': stop['stop_name'],
                'lat': float(stop['stop_lat']),
                'lon': float(stop['stop_lon']),
                'seq': int(st_row['stop_sequence'])
            })
    return result

print('✓ get_line_stops() helper loaded')

In [ ]:
# Plot function
def plot_line_map(line_name, stops_0, stops_1):
    """Create interactive map with line route and stops."""
    fig = go.Figure()
    color = LINE_COLORS.get(line_name, '#999999')
    
    # Direction 0
    if stops_0:
        lats = [s['lat'] for s in stops_0]
        lons = [s['lon'] for s in stops_0]
        
        fig.add_trace(go.Scattermapbox(
            lat=lats, lon=lons,
            mode='lines+markers',
            line=dict(width=3, color=color),
            marker=dict(size=8, color=color),
            text=[s['name'] for s in stops_0],
            hovertemplate='<b>%{text}</b><extra></extra>',
            name=f'L{line_name} Richtung A',
            legendgroup='dir0'
        ))
    
    # Direction 1 (if different)
    if stops_1 and len(stops_1) > 0:
        lats1 = [s['lat'] for s in stops_1]
        lons1 = [s['lon'] for s in stops_1]
        
        fig.add_trace(go.Scattermapbox(
            lat=lats1, lon=lons1,
            mode='lines+markers',
            line=dict(width=2, color=color, dash='dash'),
            marker=dict(size=6, color=color, opacity=0.6),
            text=[s['name'] for s in stops_1],
            hovertemplate='<b>%{text}</b><extra></extra>',
            name=f'L{line_name} Richtung B',
            legendgroup='dir1',
            visible='legendonly'
        ))
    
    fig.update_layout(
        mapbox=dict(
            style='carto-positron',
            center=dict(lat=47.378, lon=8.540),
            zoom=12
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=520,
        showlegend=True,
        title=f'Linie {line_name} — Streckenführung 2025'
    )
    return fig

print('✓ plot_line_map() helper loaded')

In [ ]:
# Demo: Linie 11
LINE = '11'
color = LINE_COLORS.get(LINE, '#999999')

print(f'\n=== Linie {LINE} ===')
print(f'Farbe: {color}\n')

stops_dir0 = get_line_stops(LINE, '0')
stops_dir1 = get_line_stops(LINE, '1')

print(f'Richtung A ({stops_dir0[0]["name"] if stops_dir0 else "?"} → {stops_dir0[-1]["name"] if stops_dir0 else "?"}):')
for i, stop in enumerate(stops_dir0, 1):
    print(f'  {i:2d}. {stop["name"]}')

if stops_dir1:
    print(f'\nRichtung B ({stops_dir1[0]["name"]} → {stops_dir1[-1]["name"]}):')
    for i, stop in enumerate(stops_dir1, 1):
        print(f'  {i:2d}. {stop["name"]}')

In [ ]:
# Show map for Linie 11
fig = plot_line_map(LINE, stops_dir0, stops_dir1)
fig.show()